# 02613 — Python and High-Performance Computing
## Exam Cheat Sheet

---

This notebook covers every recurring question type from the 2024 exam and re-exam.

**Topics:**
1. [LSF / Batch Job Scripts](#1-lsf--batch-job-scripts)
2. [Amdahl's Law & Parallelism](#2-amdahls-law--parallelism)
3. [Parallelisation Strategy](#3-parallelisation-strategy)
4. [Parallel Reduction](#4-parallel-reduction)
5. [NumPy Broadcasting](#5-numpy-broadcasting)
6. [Cache Efficiency & Memory Layout](#6-cache-efficiency--memory-layout)
7. [Profiling (cProfile & line_profiler)](#7-profiling-cprofile--line_profiler)
8. [Numba JIT](#8-numba-jit)
9. [CUDA Kernels & GPU](#9-cuda-kernels--gpu)
10. [nsys Profiler (GPU)](#10-nsys-profiler-gpu)
11. [Pandas: dtype Recoding & Indexing](#11-pandas-dtype-recoding--indexing)
12. [Zarr Chunked Arrays](#12-zarr-chunked-arrays)
13. [Chunked Processing & Memory Budget](#13-chunked-processing--memory-budget)
14. [Floating Point Precision](#14-floating-point-precision)
15. [time Command & Wall vs CPU Time](#15-time-command--wall-vs-cpu-time)
16. [Scheduling: Static vs Dynamic](#16-scheduling-static-vs-dynamic)


---
## 1. LSF / Batch Job Scripts

### Key directives

| Directive | Meaning |
|---|---|
| `#BSUB -J name` | Job name (use `name[1-N]` for job arrays) |
| `#BSUB -q hpc` | Queue (`hpc`, `gpuv100`, `gpua100`) |
| `#BSUB -W HH:MM` | Max wall-time |
| `#BSUB -n 8` | Number of CPU cores |
| `#BSUB -R "rusage[mem=XGB]"` | Memory **per core** |
| `#BSUB -R "span[hosts=1]"` | All cores on ONE node (required for shared memory) |
| `#BSUB -gpu "num=1:mode=exclusive_process"` | Request a GPU |
| `#BSUB -o file_%J.out` | Stdout (`%J` = job ID, `%I` = array index) |
| `#BSUB -w done(jobname)` | Wait until job finishes (DONE state) |
| `#BSUB -w ended(jobname)` | Wait until job ends (DONE **or** EXIT) |

### ⚠️ Memory is per core!

If your program needs **16 GB total** and you request **8 cores**:
```
#BSUB -R "rusage[mem=2GB]"   ← 16 GB / 8 cores = 2 GB per core
```

### Job array dependency

```bash
# Wait for ALL jobs to finish (even failed ones)
#BSUB -w ended(arrayjobname)

# Wait for ALL jobs to finish SUCCESSFULLY
#BSUB -w done(arrayjobname)
```
> If even one job in the array `EXIT`s, `done()` is never satisfied → the dependent job **never starts**.

### GPU job
```bash
#BSUB -q gpuv100
#BSUB -gpu "num=1:mode=exclusive_process"
```

### Job array variable
```bash
#BSUB -J myjob[1-12]
python script.py $LSB_JOBINDEX   # gives 1..12
```

In [6]:
# Amdahl's Law helpers

def amdahl_speedup(F, p):
    """Speedup with p cores and parallel fraction F."""
    return 1 / ((1 - F) + F / p)

def amdahl_max_speedup(F):
    """Theoretical maximum speedup (p -> infinity)."""
    return 1 / (1 - F)

def amdahl_F_from_speedup(S, p):
    """Estimate parallel fraction from measured speedup S on p cores."""
    return p * (1 - 1/S) / (p - 1)

def serial_time_from_parallel(T_parallel, p, F):
    """Given time on p cores, recover serial time."""
    S = amdahl_speedup(F, p)
    return T_parallel * S

# --- Examples ---
F = 0.8
print(f"Speedup with F=0.8, p=8: {amdahl_speedup(F, 8):.3f}")
print(f"Max speedup with F=0.8:  {amdahl_max_speedup(F):.1f}")

# From exam 2024 re-exam Q3: 10 min on 4 cores, F=0.8
S4 = amdahl_speedup(0.8, 4)
print(f"\nRe-exam Q3: S(4) = {S4:.2f}, serial time = {10 * S4:.1f} min")

Speedup with F=0.8, p=8: 3.333
Max speedup with F=0.8:  5.0

Re-exam Q3: S(4) = 2.50, serial time = 25.0 min


---
## 2. Amdahl's Law & Parallelism

### Formula

$$S(p) = \frac{1}{(1-F) + F/p}$$

- $F$ = parallel fraction (0–1)
- $p$ = number of processors
- Max speedup = $\frac{1}{1-F}$ (as $p \to \infty$)

### Estimating F from a speedup plot
Read the **plateau value** (where the curve flattens). That's the max speedup.
$$F = 1 - \frac{1}{\text{max speedup}}$$

E.g. plateau at 5 → $F = 1 - 1/5 = 0.8$

### Reducing serial time
If $T_{serial}$ drops by $\Delta$, the new parallel time is:
$$T'_{parallel} = T_{parallel} - \Delta$$
(The parallel part is unchanged, only the serial fraction changes.)

### Exam tip
If max speedup with best hardware < management target → **don't pursue parallelisation**.

---
## 3. Parallelisation Strategy

### Multi-threading vs Multi-processing

| | Multi-threading | Multi-processing |
|---|---|---|
| GIL blocked? | Yes (pure Python loops) | No |
| Good for | NumPy, Numba (`nogil=True`), I/O | CPU-bound pure Python |
| Overhead | Low | High (process spawn) |

> **Rule**: Pure Python CPU work → multiprocessing. NumPy/Numba (which releases GIL) → threading is fine.

### Can a loop be parallelised?
- ✅ Each iteration is **independent** of others → parallelisable
- ❌ Each iteration **depends on the previous** (e.g. simulation steps) → cannot parallelise

### Static vs Dynamic scheduling
See [Section 16](#16-scheduling-static-vs-dynamic).

### Parallel reduction
See [Section 4](#4-parallel-reduction).

In [7]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np


def process_number(n):
    s = 0
    for i in range(n):
        s += i
    return s


def sum_row(row):
    return row.sum()


# NumPy threading
x = np.random.rand(1000, 1000)

with ThreadPoolExecutor(max_workers=4) as executor:
    row_sums = list(executor.map(sum_row, x))

print("Row sums (threaded):", sum(row_sums))


# Pure Python threading
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(process_number, [10, 100, 1000]))

print("process_number results:", results)

Row sums (threaded): 500018.4291820763
process_number results: [45, 4950, 499500]


---
## 4. Parallel Reduction

A function can be used in a parallel reduction **only if it is associative** (and ideally commutative).

### Associativity check
$$f(f(a, b), c) = f(a, f(b, c)) \text{ for all inputs}$$

### Common examples

| Operation | Associative? | Commutative? | OK for reduction? |
|---|---|---|---|
| `+` (sum) | ✅ | ✅ | ✅ |
| `*` (product) | ✅ | ✅ | ✅ |
| `max`, `min` | ✅ | ✅ | ✅ |
| Set intersection `∩` | ✅ | ✅ | ✅ |
| `abs(x + y)` | ❌ | ✅ | ❌ |
| Matrix multiply | ✅ | ❌ | ✅ (order matters!) |

### Counter-example (exam Q6)
```python
def abssum(x, y):
    return abs(x + y)  # NOT associative!

# |1+2| + (-3) = 3 + (-3) → |3 + (-3)| = 0
# |2+(-3)| = |-1| = 1 → |1 + 1| = 2   ← different!
```
**Fix**: sum normally first, then take `abs` at the end.

### Parallel reduction speedup
Reduction over $n$ elements takes $\log_2(n)$ steps in parallel, vs $n$ steps serially.
$$\text{speedup} = \frac{n}{\log_2(n)}$$

---
## 5. NumPy Broadcasting

### Rules
1. Align shapes to the **right**.
2. Left-pad with `1`s until same length.
3. Dimensions are compatible if equal **or** one of them is `1`.
4. Resulting shape = element-wise max of the two shapes.

### Example
```
a: (100, 1, 6, 3)
b:      (100, 1, 3)   →  pad left → (1, 100, 1, 3)

Result: (100, 100, 6, 3)
```

### Adding dimensions with indexing
```python
a[None]       # add axis at front  → (1, ...)
a[:, None]    # add axis after dim 0
a[..., None]  # add axis at end
a[:, :, None] # add axis after dim 1
```

### Typical exam pattern: subtract mean from images
```python
images.shape      # (N, H, W, 3)
mim.shape         # (H, W)       — mean image
mean_px.shape     # (N, 3)       — mean pixel per image

images - mim[:, :, None]     # (H,W,1) broadcasts over 3 channels ✅
images - mean_px[:, None, None]  # (N,1,1,3) broadcasts over H,W ✅
```

In [8]:
import numpy as np

# Demo: subtract mean image from each channel of each image
N, H, W = 10, 64, 64
images = np.random.rand(N, H, W, 3)
mim = np.random.rand(H, W)          # mean image shape (H, W)
mean_px = np.random.rand(N, 3)      # per-image mean pixel (N, 3)

result1 = images - mim[:, :, None]     # subtracts mean image from each channel
result2 = images - mean_px[:, None, None]  # subtracts mean pixel from each spatial location

print("images shape:", images.shape)
print("mim[:,:,None] shape:", mim[:, :, None].shape, "→ result:", result1.shape)
print("mean_px[:,None,None] shape:", mean_px[:, None, None].shape, "→ result:", result2.shape)

# Broadcasting shape inference
a = np.zeros((100, 1, 6, 3))
b = np.zeros((100, 1, 3))
c = a + b
print("\na+b shape:", c.shape)  # (100, 100, 6, 3)

images shape: (10, 64, 64, 3)
mim[:,:,None] shape: (64, 64, 1) → result: (10, 64, 64, 3)
mean_px[:,None,None] shape: (10, 1, 1, 3) → result: (10, 64, 64, 3)

a+b shape: (100, 100, 6, 3)


---
## 6. Cache Efficiency & Memory Layout

### Row-wise storage (C-order, default NumPy)
The **last axis** has the smallest stride → adjacent in memory.

```
Array shape (3, 4):  stored as row0col0, row0col1, row0col2, row0col3, row1col0, ...
```

### CPU: ordering loops
**Inner-most loop should iterate over the axis with the smallest stride.**

```python
# strides = (600, 40, 8, 200) for axes (i, j, k, l)
# Smallest stride = 8 at axis k → k should be innermost
# Order from outermost to innermost: i, l, j, k

for l in range(...):
    for i in range(...):
        for j in range(...):
            for k in range(...):   # innermost = smallest stride
                x = arr[i, j, k, l]
```

### CPU conv/loop: channels last
If the inner loop is over channels → store channels **last** (`H × W × C`).

### GPU (CUDA): coalesced memory access
Threads in a **warp** execute simultaneously. For coalesced access:
- Adjacent threads (same warp) should access **adjacent memory addresses**.
- In a 2D grid `(i, j)`, threads vary **j fastest** within a warp (column-major within a block).
- So the **j axis** should have the smallest stride → **last axis** in row-major storage.

### GPU conv: channels first
In CUDA, all threads in a block process the same channel `k` simultaneously → channel values should be contiguous → channels **first** (`C × H × W`).

### Quick reference

| Context | Recommended layout |
|---|---|
| CPU parallel loop over pixels, inner loop over channels | `H × W × C` (channels last) |
| CUDA kernel, threads over pixels, inner loop over channels | `C × H × W` (channels first) |
| General CPU loops | Innermost loop = smallest stride axis |

In [9]:
import numpy as np

# Inspecting strides
a = np.zeros((5, 10, 3), dtype=np.float32)  # H x W x C
print("H x W x C strides:", a.strides)   # last axis = smallest

b = np.zeros((3, 5, 10), dtype=np.float32)  # C x H x W
print("C x H x W strides:", b.strides)   # last axis = smallest

# Reshape: a.reshape(-1)[i] — row-major flattening
c = np.array([[1,5,43,51,32],[73,2,4,67,37],[9,3,54,8,22]])
print("\nFlattened:", c.reshape(-1))
print("Index 8:", c.reshape(-1)[8])   # 67

H x W x C strides: (120, 12, 4)
C x H x W strides: (200, 40, 4)

Flattened: [ 1  5 43 51 32 73  2  4 67 37  9  3 54  8 22]
Index 8: 67


---
## 7. Profiling (cProfile & line_profiler)

### cProfile output columns

| Column | Meaning |
|---|---|
| `ncalls` | Number of times the function was called |
| `tottime` | Time spent in the function itself (excluding callees) |
| `percall` | tottime / ncalls |
| `cumtime` | Total time including all called sub-functions |

### Reading ncalls to infer data size
If `process_sample` has `ncalls = 10`, there were **10 samples** in the profiling subset.

### Identifying bottleneck for production workload
Scale `percall` by the **production call count**, not the profiling count:
```
process_sample: percall = 0.505 s, production calls = 1000
→ estimated production time = 505 s

load_params: cumtime = 20 s (called once, does not scale with samples)
```
→ Focus on `process_sample`.

### line_profiler output columns

| Column | Meaning |
|---|---|
| `Hits` | How many times that line executed |
| `Time` | Total time for that line (in timer units, often µs) |
| `Per Hit` | Time / Hits |
| `% Time` | Fraction of total function time |

### FLOP/s from line profiler
1. Count FLOPs per loop iteration (each `+`, `-`, `*`, `/`, `sqrt` = 1 FLOP).
2. Multiply by number of iterations (`Hits`).
3. Divide by total time (convert to seconds).

```
a = x[i]*x[i] + 4      → 2 FLOPs
b = y[n-i-1] / x[i]   → 1 FLOP
z = z + a / b          → 2 FLOPs
Total: 5 FLOPs/iter × 10000 iters = 50000 FLOPs
Total time = 15000 µs = 0.015 s
→ 50000 / 0.015 = 3.33 × 10^6 FLOP/s
```

### Estimating production runtime
```
prep_conds: 2.0 s (called once → constant)
process_single: 1.27 s for 1000 items → 12.7 s for 10000 items
Total for 10000 items: 2.0 + 12.7 = 14.7 s
```

In [10]:
# Run cProfile on any function
import cProfile
import pstats
import io

def example_function():
    return sum(i*i for i in range(100000))

pr = cProfile.Profile()
pr.enable()
example_function()
pr.disable()

s = io.StringIO()
ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
ps.print_stats(10)
print(s.getvalue())

# Install line_profiler: pip install line_profiler
# Usage:
#   @profile
#   def my_func(): ...
#   Then run: kernprof -l -v script.py

         100256 function calls (100254 primitive calls) in 0.015 seconds

   Ordered by: cumulative time
   List reduced from 84 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.012    0.012 /Users/dittegilsfeldt/GitHub_DTU/MSc_Human-Centered_Artificial_Intelligence_DTU/.venv/lib/python3.13/site-packages/decorator.py:232(fun)
        1    0.000    0.000    0.012    0.012 /Users/dittegilsfeldt/GitHub_DTU/MSc_Human-Centered_Artificial_Intelligence_DTU/.venv/lib/python3.13/site-packages/IPython/core/history.py:94(only_when_enabled)
        1    0.000    0.000    0.012    0.012 /Users/dittegilsfeldt/GitHub_DTU/MSc_Human-Centered_Artificial_Intelligence_DTU/.venv/lib/python3.13/site-packages/IPython/core/history.py:1047(writeout_cache)
        1    0.000    0.000    0.012    0.012 /Users/dittegilsfeldt/GitHub_DTU/MSc_Human-Centered_Artificial_Intelligence_DTU/.venv/lib/python3.13/site-packages/IPython/cor

---
## 8. Numba JIT

### Basic usage
```python
from numba import jit

@jit(nopython=True)           # compile to machine code, no Python fallback
def fast_function(x):
    ...

@jit(nopython=True, nogil=True)  # also releases the GIL → can use threading
def parallelisable_function(x):
    ...
```

### When to use threading vs processing with Numba
- `nogil=True` → function releases the GIL when running → **multi-threading works**.
- Without `nogil=True` → GIL is held → use **multi-processing**.

### Key rules
- Numba compiles on **first call** (warm-up cost).
- Works best with loops over arrays (replaces NumPy for loop-heavy code).
- Cannot parallelise a loop where each step depends on the previous.

### Example from exam (Q19)
```python
@jit(nopython=True, nogil=True)
def simulate_single(x0, n, step):  # loop is sequential → can't parallelise here
    x = x0
    for i in range(n):
        x = x + step * np.cos(x) / (np.sin(5*x) + 2.5)
    return x

# Parallelise the OUTER loop (independent initial conditions)
from multiprocessing.pool import ThreadPool  # threading OK because nogil=True

def simulate(n, x0s, step):
    with ThreadPool(len(x0s)) as p:
        return p.starmap(simulate_single, [(x0, n, step) for x0 in x0s])
```

---
## 9. CUDA Kernels & GPU

### Kernel structure
```python
from numba import cuda

@cuda.jit
def my_kernel(array, out):
    i, j = cuda.grid(2)          # 2D thread index
    if i < array.shape[0] and j < array.shape[1]:
        out[i, j] = array[i, j] * 2

# Launch
threadsperblock = (32, 32)
blockspergrid = (
    (H + 31) // 32,
    (W + 31) // 32
)
my_kernel[blockspergrid, threadsperblock](array, out)
```

### Number of thread blocks
$$\text{blocks per dim} = \lceil \text{size} / \text{threads per block} \rceil$$

Example: 200×200 output, 16×16 blocks → $\lceil 200/16 \rceil = 13$ → **13×13 blocks**.

### Warp & coalesced access
- A **warp** = 32 threads that execute simultaneously.
- Threads in a warp are indexed along the **first dimension of the block** (x-axis, i.e. columns in a 2D block).
- For coalesced access: threads in the same warp should access **consecutive memory** → the axis varying fastest within a warp should be the **last axis** (smallest stride).

### Thread block configuration for 2D kernels

| Config | Warp alignment | Performance |
|---|---|---|
| `(1, 256)` | 256 threads along column (j) → good if j is last axis | ✅ |
| `(256, 1)` | 256 threads along row (i) → strided access if j is last | ❌ |
| `(16, 16)` | Mixed — generally fine for 2D | ✅ |

### Avoiding excess memory transfers
NumPy arrays passed to a CUDA kernel → **auto transferred to/from GPU every call**.

```python
# Inefficient: NumPy arrays → 2 HtoD + 2 DtoH per kernel call
average3x3[blocks, threads](x_numpy, y_numpy, n)

# Efficient: allocate y on GPU; only transfer x in, y out once
y_gpu = cuda.to_device(np.zeros(...))
for x in x_all:
    x_gpu = cuda.to_device(x)
    average3x3[blocks, threads](x_gpu, y_gpu, n)
y = y_gpu.copy_to_host()   # single DtoH at the end
```

---
## 10. nsys Profiler (GPU)

### Reading the output

```
** CUDA GPU Kernel Summary:
Time(%)  Total Time(s)  Name
100.0    0.5            my_kernel

** GPU MemOps Summary (by Time):
Time(%)  Total Time(s)  Operation
83.3     2.5            [CUDA memcpy HtoD]   ← CPU→GPU
16.7     0.5            [CUDA memcpy DtoH]   ← GPU→CPU

** GPU MemOps Summary (by Size):
Total(MB)  Operation
25000      [CUDA memcpy HtoD]
1000       [CUDA memcpy DtoH]
```

### Transfer speed calculation
$$\text{speed} = \frac{\text{Total MB}}{\text{Total Time (s)}}$$

But note: **one transfer might be 0 MB** (check the Max column).

Example:
- HtoD: 25000 MB total, but count=2 and Med=12500 MB → one transfer is ~25 GB, one is ~0 GB.
- Actual data: 25000 MB in 2.5 s → ~**10 GB/s**.

### Total GPU pipeline time
$$T_{GPU} = T_{kernel} + T_{HtoD} + T_{DtoH}$$

Example: 0.5 + 2.5 + 0.5 = **3.5 s** → 2× faster than CPU at 7 s.

### What takes the most time?
Compare `Total Time` across kernel summary and MemOps summary:
- Often **memory transfers dominate** → optimise by keeping data on GPU across calls.

---
## 11. Pandas: dtype Recoding & Indexing

### dtype recoding strategy

| Column type | Strategy |
|---|---|
| Dates as strings | → `datetime64` |
| Strings with few unique values (< ~255) | → `category` (or `uint8` integer code) |
| Integers with small range | → smallest int type that fits the range |
| Floats with limited precision needed | → `float32` instead of `float64` |

### Integer type ranges

| dtype | Min | Max |
|---|---|---|
| `int8` | -128 | 127 |
| `uint8` | 0 | 255 |
| `int16` | -32768 | 32767 |
| `uint16` | 0 | 65535 |
| `int32` | -2.1B | 2.1B |
| `uint32` | 0 | 4.3B |
| `int64` | -9.2e18 | 9.2e18 |

### Speeding up date-based row extraction
```python
# Set date as sorted index → O(log n) lookup instead of O(n) scan
df = df.set_index('date').sort_index()
rows = df.loc['2024-05-28']   # fast!
```
Overhead of building the index is worth it when **many queries** are performed.

In [11]:
import pandas as pd
import numpy as np

# Example: recode dtypes to reduce memory
df = pd.DataFrame({
    'date': pd.date_range('2020-01-01', periods=1000),
    'location': np.random.choice(['Lyngby', 'Copenhagen', 'Aarhus'], 1000),
    'mach_id': np.random.randint(-1, 5731, 1000),
    'units': np.random.randint(932, 68838, 1000),
})

print("Before:", df.memory_usage(deep=True).sum() / 1e6, "MB")

df['location'] = df['location'].astype('category')
df['mach_id'] = df['mach_id'].astype('int16')   # fits in -32768..32767
df['units'] = df['units'].astype('int32')        # fits in 32-bit

print("After: ", df.memory_usage(deep=True).sum() / 1e6, "MB")

# Set index for fast date lookups
df = df.set_index('date').sort_index()
print("\nIndex type:", df.index.dtype)

Before: 0.080436 MB
After:  0.015301 MB

Index type: datetime64[us]


---
## 12. Zarr Chunked Arrays

### Key idea
Zarr reads data from disk in **chunks**. Choose chunk shape to minimise the number of chunk reads for your access pattern.

### Access pattern → chunk shape

| You read... | Best chunk shape |
|---|---|
| Full rows (e.g. `x[i, :]`) | Wide chunks: `(1, ncols)` |
| Full columns (e.g. `x[:, j]`) | Tall chunks: `(nrows, 1)` or `(nrows, small)` |
| Sub-blocks | Square-ish chunks |

### Example (re-exam Q5)
Matrix `1000 × 100000`, accessing full columns:
- Option (c) `1000 × 100` — entire column fits in one chunk → **1 read per column** ✅
- Option (a) `10 × 10000` — need 100 reads per column ❌

### Memory per chunk
```
chunk shape × dtype bytes
e.g. (1000, 100) × float64 (8 bytes) = 800,000 bytes = 800 KB
```

### Zarr chunk size for loop over rows (Q20)
```python
for i in range(a.shape[0]):
    s[i] = np.sum(a[i])     # reads one row at a time
# Best chunk: (1, ncols) → each iteration loads exactly 1 chunk
```

---
## 13. Chunked Processing & Memory Budget

### Calculating max chunk size

$$\text{rows per chunk} = \frac{\text{available memory (bytes)}}{\text{bytes per row}}$$

$$\text{bytes per row} = \sum_{\text{columns}} \text{dtype size (bytes)}$$

### dtype sizes

| dtype | Bytes |
|---|---|
| `uint8` / `int8` | 1 |
| `uint16` / `int16` | 2 |
| `uint32` / `int32` / `float32` | 4 |
| `uint64` / `int64` / `float64` | 8 |

### Example (exam 2024 Q17)
```
sensor_id: uint32 = 4 bytes
timestamp: uint64 = 8 bytes
power:     float64 = 8 bytes
→ 20 bytes per row

200 MB = 200 × 10^6 bytes
max rows = 200,000,000 / 20 = 10,000,000  → ~10 million rows ✅
```

### Re-exam Q18 (24 MB, 3 × int64 columns)
```
bytes per row = 3 × 8 = 24
max rows = 24 × 10^6 / 24 = 1,000,000
```

In [12]:
def max_chunk_rows(available_mb, col_dtypes):
    """
    col_dtypes: list of numpy dtypes, e.g. [np.uint32, np.uint64, np.float64]
    available_mb: memory budget in MB
    """
    import numpy as np
    bytes_per_row = sum(np.dtype(dt).itemsize for dt in col_dtypes)
    available_bytes = available_mb * 1_000_000
    return int(available_bytes / bytes_per_row)

import numpy as np

# Exam 2024 Q17: sensor DataFrame
rows = max_chunk_rows(200, [np.uint32, np.uint64, np.float64])
print(f"Exam Q17 max rows: {rows:,}")   # ~10,000,000

# Re-exam Q18: 3 × int64
rows2 = max_chunk_rows(24, [np.int64, np.int64, np.int64])
print(f"Re-exam Q18 max rows: {rows2:,}")  # ~1,000,000

Exam Q17 max rows: 10,000,000
Re-exam Q18 max rows: 1,000,000


---
## 14. Floating Point Precision

### float16 characteristics
- Range: approx ±65504
- Resolution: ~0.001 relative
- Precision: ~3 significant decimal digits

### Key concept: relative precision
The absolute precision depends on the **magnitude** of the number:
$$\text{absolute precision} \approx \text{value} \times \text{resolution}$$

```
10000 + 1:
  absolute precision at 10000 = 10000 × 0.001 = 10
  So 10001 cannot be represented (gap is 10, but we need gap of 1)
  → rounds to 10000
```

### General rule
- `float16`: ~3 significant digits → careful for large numbers
- `float32`: ~7 significant digits
- `float64`: ~15 significant digits

In [13]:
import numpy as np

print("float16 info:", np.finfo(np.float16))
print()

a = np.array(10000, dtype='float16')
b = np.array(1, dtype='float16')
print("10000 + 1 in float16:", a + b)   # prints 10000.0

a32 = np.array(10000, dtype='float32')
b32 = np.array(1, dtype='float32')
print("10000 + 1 in float32:", a32 + b32)   # prints 10001.0

float16 info: Machine parameters for float16
---------------------------------------------------------------
precision = 3   resolution = 0.001
machep = -10   eps =        0.000977
negep =  -11   epsneg =     0.0004883
minexp = -14   tiny =       6.104e-05
maxexp = 16   max =        6.55e+04
nexp =   5   min =        -max
smallest_normal = 6.104e-05   smallest_subnormal = 6e-08
---------------------------------------------------------------


10000 + 1 in float16: 1e+04
10000 + 1 in float32: 10001.0


---
## 15. `time` Command & Wall vs CPU Time

### Output
```
real  0m12.03s   ← wall-clock time (what you actually wait)
user  0m12.00s   ← CPU time in user mode
sys   0m0.03s    ← CPU time in kernel mode
```

### Parallelism effect
| | Single core | 2 cores (perfect parallelism) |
|---|---|---|
| `real` (wall) | 12 s | **6 s** (halved) |
| `user` (CPU) | 12 s | **12 s** (unchanged — summed over cores) |

> **Rule**: `real` goes down with more cores. `user` stays the same (or goes up slightly due to overhead).

---
## 16. Scheduling: Static vs Dynamic

### Static scheduling
- Tasks divided evenly **upfront** across workers.
- Best when all tasks take **roughly the same time** (low variance).
- Low overhead.

### Dynamic scheduling
- Workers pick up new tasks from a queue as they finish.
- Best when task durations vary a lot (**high variance**).
- Higher overhead (queue management), but prevents workers sitting idle.

### How to tell from profiler output

| Metric | Verdict |
|---|---|
| `StdDev` ≈ 0 (small relative to `Avg`) | Static is fine |
| `StdDev` ≈ 2× `Avg` or larger | Use dynamic scheduling |

### Exam example (F25 Q10)
```
kernel1: Avg=20ms, StdDev=40ms → high variance → use DYNAMIC
kernel2: Avg=20ms, StdDev=0.05ms → stable → STATIC is fine
```

### Python APIs
```python
from multiprocessing.pool import Pool, ThreadPool

# Static: divide work equally upfront
p.map(func, tasks, chunksize=len(tasks)//n_workers)

# Dynamic: chunksize=1 → one task at a time (most dynamic)
p.map(func, tasks, chunksize=1)

# Or use imap_unordered for streaming dynamic scheduling
for result in p.imap_unordered(func, tasks, chunksize=1):
    process(result)
```

---
## Quick Decision Guide

```
Slow code?
│
├─ Profile first (cProfile / line_profiler / nsys)
│   └─ Find the bottleneck, scale by production workload
│
├─ CPU-bound pure Python loop?
│   ├─ Independent iterations → multiprocessing (or Numba @jit)
│   └─ Dependent iterations → can't parallelise, use Numba
│
├─ NumPy / Numba (nogil=True) ?
│   └─ Independent iterations → multithreading OK
│
├─ Task durations vary a lot?
│   └─ Use dynamic scheduling
│
├─ Memory layout wrong?
│   ├─ CPU: inner loop axis = smallest stride
│   └─ GPU: warp threads access adjacent memory (last axis = smallest stride)
│
├─ GPU memory transfers dominating?
│   └─ Allocate output on GPU, transfer input once, copy result back once
│
└─ Pandas too slow/large?
    ├─ Recode dtypes (category, smaller ints, datetime)
    ├─ Set sorted index for fast lookups
    └─ Process in chunks if too large for memory
```